In [ ]:
import pandas as pd
import os

from core.visualization.kaplan_meier import plot_kaplan_meier_curves
from core.utils import load_predictions
from core.utils import save_and_show


# **Load data**

In [ ]:
from config import (
    USE_DUMMY_DATA,
    BM_RUN_PATH, PB_RUN_PATH,
    SURVIVAL_DATA_PATH_BM, SURVIVAL_DATA_PATH_PB,
)

MODEL_PREDICTIONS_PATH_BM = BM_RUN_PATH
MODEL_PREDICTIONS_PATH_PB = PB_RUN_PATH


### Bone marrow

In [ ]:
df_survival_bm = pd.read_excel(SURVIVAL_DATA_PATH_BM)

### Peripheral blood

In [ ]:
df_survival_pb = pd.read_excel(SURVIVAL_DATA_PATH_PB)

# **Overall survival**

In [ ]:
# we can use either BM or PB data because same patients
df = df_survival_bm.copy()

max_time = df["time_dg_to_death"].max()
df["time_dg_to_death"] = (
    df["time_dg_to_death"]
    .fillna(max_time) / 30
)

# treatment groups
risk_cols = ["cyto", "aza", "aza_ven", "palliative"]

fig = plot_kaplan_meier_curves(
    df=df,
    time_col="time_dg_to_death",
    event_col="died",
    group_cols=risk_cols,
    title="Overall Survival by Risk Group",
    xlim=(0, 96),
    figsize=(8, 6),
    fontsize=14,
    xlabel="Time (months)"
)
save_and_show(fig, figure_type="main")

# **Survival based on MIL predictions**

In [ ]:
def _extract_variable_from_folder(folder_name: str) -> str:
    """Extract the variable portion from a folder name."""
    RUN_PREFIX = "run_"
    FOLDER_SPLIT_DELIMITER = "_"
    FOLDER_VARIABLE_SPLIT_INDEX = 3  # extract everything after this index

    if RUN_PREFIX in folder_name:
        # Extract everything after the Nth underscore
        parts = folder_name.split(FOLDER_SPLIT_DELIMITER, FOLDER_VARIABLE_SPLIT_INDEX)
        return parts[-1]
    return folder_name


def collect_performance_metrics(base_dir: str, metric: str) -> pd.DataFrame:

    def _read_metric_values(df: pd.DataFrame, metric: str) -> tuple[float, float]:
        """Extract mean and std metric values from the DataFrame."""

        FOLD_COLUMN = "fold"
        FOLD_MEAN_LABEL = "mean"
        FOLD_STD_LABEL = "std"

        mean_val = df.loc[df[FOLD_COLUMN] == FOLD_MEAN_LABEL, metric].values[0]
        std_val = df.loc[df[FOLD_COLUMN] == FOLD_STD_LABEL, metric].values[0]
        return mean_val, std_val

    RESULT_FILE_NAME = "results.xlsx"
    file_path = os.path.join(base_dir, RESULT_FILE_NAME)
    df = pd.read_excel(file_path)
    mean_val, std_val = _read_metric_values(df, metric)

    return mean_val, std_val


def split_label(text):
    """
    Split a long string into two lines near the middle.
    Returns the original string if no space is found.
    """
    if not text or " " not in text:
        return text
    
    mid = len(text) // 2
    # find nearest space to the midpoint
    left_space = text.rfind(" ", 0, mid)
    right_space = text.find(" ", mid)

    # choose the closer space
    if left_space == -1:
        split_pos = right_space
    elif right_space == -1:
        split_pos = left_space
    else:
        split_pos = left_space if (mid - left_space) < (right_space - mid) else right_space

    return text[:split_pos] + "\n" + text[split_pos+1:]


def extract_prediction_results(requested_variable, prediction_result_dir):

    for run_name in os.listdir(prediction_result_dir):
        run_dir = os.path.join(prediction_result_dir, run_name)
        variable_name = _extract_variable_from_folder(run_name)
        if variable_name == requested_variable:
            all_results = []
            for item in os.listdir(run_dir):
                sub_dir = os.path.join(run_dir, item)
                if os.path.isdir(sub_dir):
                    fold_num = int(sub_dir.split("_")[-1])
                    result_path = os.path.join(sub_dir, f"cell_attention_scores_fold={fold_num}.csv")
                    results = pd.read_csv(result_path)
                    results.drop_duplicates(subset="bag_id", inplace=True)
                    results = results[["album_id", "database_id", "true_label", "pred_label", "pred_prob"]]
                    all_results.append(results)
            df = pd.concat(all_results)

            df["true_positive"] = (df["true_label"] == 1) & (df["pred_label"] == 1)
            df["false_positive"] = (df["true_label"] == 0) & (df["pred_label"] == 1)
            df["true_negative"] = (df["true_label"] == 0) & (df["pred_label"] == 0)
            df["false_negative"] = (df["true_label"] == 1) & (df["pred_label"] == 0)
            df["pred_positive"] = df["pred_label"].astype(bool)
            df["pred_negative"] = ~df["pred_positive"]
            df["target_positive"] = df["true_label"].astype(bool)
            df["target_negative"] = ~df["target_positive"]

            return df

In [ ]:
def plot_all_experiments(df_survival, prediction_results_dir, treatment_groups, pairs_to_test):

    for run_name in os.listdir(prediction_results_dir):

        run_dir = os.path.join(prediction_results_dir, run_name)
        variable_name = _extract_variable_from_folder(run_name)

        # extract prediction results
        prediction_results = extract_prediction_results(requested_variable=variable_name, prediction_result_dir=prediction_results_dir)

        # merge data
        df_variable = df_survival.merge(prediction_results, on=["album_id", "database_id"], how="inner")

        # confert time to months
        max_time = df_variable["time_dg_to_death"].max()
        df_variable["time_dg_to_death"] = (
            df_variable["time_dg_to_death"]
            .fillna(max_time) / 30
        )

        for treatment_group in treatment_groups:

            # get specific treatment group
            df = df_variable[df_variable[treatment_group] == True]

            print(f"Variable: {variable_name}, Treatment group: {treatment_group}")

            if "TARGETS" in pairs_to_test:
                print("Experiment: Targets")
                risk_cols = ["target_positive", "target_negative"]
                fig = plot_kaplan_meier_curves(
                    df=df,
                    time_col="time_dg_to_death",
                    event_col="died",
                    group_cols=risk_cols,
                    title=variable_name,
                    xlim=(0, 96),
                    figsize=(5, 3),
                    fontsize=12,
                    xlabel="Time (months)",
                    max_p_value_to_plot=0.05
                )
                save_and_show(fig, figure_type="main")

            if "PREDICTED" in pairs_to_test:
                print("Experiment: Predicted")
                risk_cols = ["pred_positive", "pred_negative"]
                fig = plot_kaplan_meier_curves(
                    df=df,
                    time_col="time_dg_to_death",
                    event_col="died",
                    group_cols=risk_cols,
                    title=variable_name,
                    xlim=(0, 96),
                    figsize=(5, 3),
                    fontsize=12,
                    xlabel="Time (months)",
                    max_p_value_to_plot=0.05
                )
                save_and_show(fig, figure_type="main")

            if "FP_VS_TN" in pairs_to_test:
                print("Experiment: False positive vs. True negative")
                risk_cols = ["false_positive", "true_negative"]
                df_ = df[df["true_label"] == False]
                fig = plot_kaplan_meier_curves(
                    df=df_,
                    time_col="time_dg_to_death",
                    event_col="died",
                    group_cols=risk_cols,
                    title=variable_name,
                    xlim=(0, 96),
                    figsize=(5, 3),
                    fontsize=12,
                    xlabel="Time (months)",
                    max_p_value_to_plot=0.05
                )
                save_and_show(fig, figure_type="main")

            if "FN_VS_TP" in pairs_to_test:
                print("False negative vs. True positive")
                risk_cols = ["false_negative", "true_positive"]
                df_ = df[df["true_label"] == True]
                if len(df_) > 0:
                    fig = plot_kaplan_meier_curves(
                        df=df_,
                        time_col="time_dg_to_death",
                        event_col="died",
                        group_cols=risk_cols,
                        title=variable_name,
                        xlim=(0, 96),
                        figsize=(5, 3),
                        fontsize=12,
                        xlabel="Time (months)",
                        max_p_value_to_plot=0.05
                    )
                    save_and_show(fig, figure_type="main")

        print("------------------------------------------------------------------------------------------------------------")

In [ ]:
PAIRS_TO_TEST = [
    "TARGETS", # actual labels for baseline
    #"PREDICTED", # predicted labels
    "FP_VS_TN", # targets are wild-types/normal, but some were predicted as mutated/abnormal
    #"FN_VS_TP", # targets are mutated/abnormal, but some were predicted as wild-types/normal
]
TREATMENT_GROUPS = [
    "cyto", 
    "aza", 
    "aza_ven", 
    "palliative",
]

### Bone marrow

In [ ]:
df_survival_bm.head()

In [ ]:
plot_all_experiments(df_survival_bm, PREDICTION_RESULTS_DIR_BM, TREATMENT_GROUPS, PAIRS_TO_TEST)

In [ ]:
for run_name in os.listdir(PREDICTION_RESULTS_DIR_BM):

    run_dir = os.path.join(PREDICTION_RESULTS_DIR_BM, run_name)
    variable_name = _extract_variable_from_folder(run_name)

    # extract prediction results
    prediction_results = extract_prediction_results(requested_variable=variable_name, prediction_result_dir=PREDICTION_RESULTS_DIR_BM)

    # merge data
    df_variable = df_survival_bm.merge(prediction_results, on=["album_id", "database_id"], how="inner")

    # confert time to months
    max_time = df_variable["time_dg_to_death"].max()
    df_variable["time_dg_to_death"] = (
        df_variable["time_dg_to_death"]
        .fillna(max_time) / 30
    )

    for treatment_group in TREATMENT_GROUPS:

        # get specific treatment group
        df = df_variable[df_variable[treatment_group] == True]

        print(f"Variable: {variable_name}, Treatment group: {treatment_group}")

        if "TARGETS" in PAIRS_TO_TEST:
            risk_cols = ["target_positive", "target_negative"]
            fig = plot_kaplan_meier_curves(
                df=df,
                time_col="time_dg_to_death",
                event_col="died",
                group_cols=risk_cols,
                title=variable_name,
                xlim=(0, 96),
                figsize=(5, 3),
                fontsize=12,
                xlabel="Time (months)",
                max_p_value_to_plot=0.05
            )
            save_and_show(fig, figure_type="main")

        if "PREDICTED" in PAIRS_TO_TEST:
            risk_cols = ["pred_positive", "pred_negative"]
            fig = plot_kaplan_meier_curves(
                df=df,
                time_col="time_dg_to_death",
                event_col="died",
                group_cols=risk_cols,
                title=variable_name,
                xlim=(0, 96),
                figsize=(5, 3),
                fontsize=12,
                xlabel="Time (months)",
                max_p_value_to_plot=0.05
            )
            save_and_show(fig, figure_type="main")

        if "FP_VS_TN" in PAIRS_TO_TEST:
            risk_cols = ["false_positive", "true_negative"]
            df_ = df[df["true_label"] == False]
            fig = plot_kaplan_meier_curves(
                df=df_,
                time_col="time_dg_to_death",
                event_col="died",
                group_cols=risk_cols,
                title=variable_name,
                xlim=(0, 96),
                figsize=(5, 3),
                fontsize=12,
                xlabel="Time (months)",
                max_p_value_to_plot=0.05
            )
            save_and_show(fig, figure_type="main")

        if "FN_VS_TP" in PAIRS_TO_TEST:
            risk_cols = ["false_negative", "true_positive"]
            df_ = df[df["true_label"] == True]
            if len(df_) > 0:
                fig = plot_kaplan_meier_curves(
                    df=df_,
                    time_col="time_dg_to_death",
                    event_col="died",
                    group_cols=risk_cols,
                    title=variable_name,
                    xlim=(0, 96),
                    figsize=(5, 3),
                    fontsize=12,
                    xlabel="Time (months)",
                    max_p_value_to_plot=0.05
                )
                save_and_show(fig, figure_type="main")

    print("------------------------------------------------------------------------------------------------------------")

### Peripheral blood

In [ ]:
plot_all_experiments(df_survival_pb, PREDICTION_RESULTS_DIR_PB, TREATMENT_GROUPS, PAIRS_TO_TEST)